# Record the app from instructions (browser-use)

You say what to show in plain words, and [browser-use](https://github.com/browser-use/browser-use)'s agent does it in Chromium while the screen is recorded. At each step the model gets the page (its elements and a screenshot) and picks the next clicks and typing. It needs no selectors, and it copes when the page changes.

You get two videos, each with its steps:

- `<name>.mp4`: the whole run in real time, including the seconds the model spends thinking between steps (the page sits still).
- `<name>-tight.mp4`: the same without those pauses. Each step runs from just before its actions to when the page settles, typing plays 4× as fast, and it zooms in on each click (1.5×) and back out. While the agent waits (30 s at a time while makable builds), only what changes on the page is kept, from half a second before each change to a second after.
- `<name>.steps.json` for each: every step's goal, what it did ("Clicked button Add"), when, in seconds into that video, and where it clicked. The Shorts notebook writes the script from these (section 2): each scene says what's on screen over the stretch of the recording it's about.

**Before you start**

- A **CPU** runtime is enough (no GPU).
- This is a separate notebook because browser-use pins its own `google-genai` and `numpy`, and the Shorts notebook (Chatterbox, Omni) can't use those.
- The model is Gemini on **Agent Platform (Vertex AI)**, with the same project and sign-in as the Shorts notebook. Or set `USE_VERTEX = False` and add the `GEMINI_API_KEY` secret. Each step sends the page and a screenshot (several thousand tokens), so a 10-step recording costs cents on Flash.
- The page has to be reachable from Colab. Section 3 records makable signed in, as a demo user with a password instead of GitHub (set up once: see the README's *Recording makable signed in*).

## Setup

browser-use pins versions of packages Colab already has, so pip lists conflicts with Colab's own (`google-colab`, `bigframes`, `google-adk`, `huggingface-hub`). That's expected; this notebook doesn't use those. After the install, restart the session (*Runtime → Restart session*) so no old versions stay loaded, then go on from the next cell.

In [ ]:
!pip install -q browser-use==0.13.11 playwright
!playwright install --with-deps chromium > /dev/null  # browser-use runs Playwright's Chromium


In [ ]:
import os
from pathlib import Path

from google.colab import auth, drive, userdata

USE_VERTEX = True  # Agent Platform (Vertex AI), paid from Cloud credits; False: an AI Studio key
PROJECT = "project-00523b00-4ba9-44e7-9e2"
MODEL = "gemini-3.6-flash"  # ~2 s a call on Vertex; 3.8 Flash queued for 5-60 s there (Oct 2026)
FALLBACK_MODEL = "gemini-3-flash-preview"  # used for the rest of a run when MODEL answers 429 or 5xx

os.environ["ANONYMIZED_TELEMETRY"] = "false"  # else browser-use reports usage (and syncs runs to its cloud)
os.environ["TIMEOUT_TypeTextEvent"] = "180"  # it types a key at a time: a long prompt can take more than its 60 s
drive.mount("/content/drive")
if USE_VERTEX:
    auth.authenticate_user()

OUT = Path("/content/drive/MyDrive/makable-shorts/recordings")


In [ ]:
os.makedirs("/content/shorts", exist_ok=True)


In [ ]:
%%writefile /content/shorts/__init__.py
"""makable Shorts: a 1-minute vertical video about what changed in makable, made by stages that share a folder.

Each stage is a program of its own with its own packages (`shorts.stages`), reading and writing a video's run
folder under the names in `shorts.run`, the contract between them: record (the footage) -> write-script
(Gemini) -> film-intro (Gemini Omni) -> revoice-intro and speak (your voice, on a GPU) -> assemble (ffmpeg).
`shorts.orchestrator` runs what's due, each stage in a process of its own. The other modules are what the
stages use, and know nothing of run folders.
"""


In [ ]:
%%writefile /content/shorts/media.py
"""Small wrappers around ffmpeg and ffprobe."""

import json
import re
import subprocess
from pathlib import Path


def ffmpeg(*args: str | Path) -> None:
    """Runs ffmpeg (overwriting outputs) and raises with its error text if it fails."""
    command = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", *map(str, args)]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed: {result.stderr.strip()}")


def duration(path: Path) -> float:
    """Returns a media file's length in seconds."""
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "json", str(path)],
        capture_output=True,
        text=True,
        check=True,
    )
    return float(json.loads(result.stdout)["format"]["duration"])


def frame_size(path: Path) -> tuple[int, int]:
    """Returns a video's width and height."""
    result = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-select_streams",
            "v:0",
            "-show_entries",
            "stream=width,height",
            "-of",
            "json",
            str(path),
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    stream = json.loads(result.stdout)["streams"][0]
    return stream["width"], stream["height"]


def has_sound(path: Path) -> bool:
    """Whether a media file has an audio stream."""
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=index", "-of", "json", str(path)],
        capture_output=True,
        text=True,
        check=True,
    )
    return bool(json.loads(result.stdout).get("streams"))


def changes(path: Path, fps: int, pixel: int = 8, area: float = 0.15) -> list[float]:
    """Returns when a video's picture changes (seconds into it): each frame in which more than `area` % of
    the pixels got more than `pixel` (of 255) lighter or darker since the frame before."""
    # The changed pixels, averaged over a 45x80 grid (16x16-pixel cells of a 720x1280 frame): ffmpeg does
    # it 8x faster than signalstats. accurate_rnd, or a cell with nothing changed comes out 1, not 0.
    cells = 45 * 80
    result = subprocess.run(
        [
            "ffmpeg",
            "-hide_banner",
            "-loglevel",
            "error",
            "-i",
            str(path),
            "-an",
            "-vf",
            f"format=gray,tblend=all_mode=difference,lutyuv=y='if(gt(val,{pixel}),255,0)',"
            "scale=45:80:flags=area+accurate_rnd",
            "-f",
            "rawvideo",
            "-pix_fmt",
            "gray",
            "-",
        ],
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed: {result.stderr.decode().strip()}")
    grids = result.stdout
    return [
        (i // cells + 1) / fps  # tblend's frame n is how frame n + 1 differs from frame n
        for i in range(0, len(grids), cells)
        if sum(grids[i : i + cells]) / (255 * cells) * 100 > area
    ]


def speech_span(path: Path, noise_db: int = -35, min_silence: float = 0.25) -> tuple[float, float]:
    """Returns when the speech in an audio file starts and ends, skipping silence at either end."""
    result = subprocess.run(
        [
            "ffmpeg",
            "-hide_banner",
            "-i",
            str(path),
            "-af",
            f"silencedetect=noise={noise_db}dB:d={min_silence}",
            "-f",
            "null",
            "-",
        ],
        capture_output=True,
        text=True,
    )
    total = duration(path)
    starts = [float(s) for s in re.findall(r"silence_start: (-?[\d.]+)", result.stderr)]
    ends = [float(s) for s in re.findall(r"silence_end: (-?[\d.]+)", result.stderr)]
    silences = list(zip(starts, ends + [total] * (len(starts) - len(ends)), strict=False))
    begin = silences[0][1] if silences and silences[0][0] <= 0.05 else 0.0
    end = silences[-1][0] if silences and silences[-1][1] >= total - 0.05 else total
    return (begin, end) if end > begin else (0.0, total)


In [ ]:
%%writefile /content/shorts/browse.py
"""Footage from instructions: browser-use's agent does a task in Chromium while the screen is recorded.

At each step browser-use shows the model the page (its elements and a screenshot) and runs the
actions the model picks, so you write what to show, not the clicks. Each step's goal and actions
are kept with when they happened, so the script can say what's on screen at each moment.

browser-use's own recorder (`record_video_dir`) writes Chrome's screencast frames as they arrive,
at a fixed rate. Chrome only sends a frame when the page changes, so that video runs fast and skips
pauses (27 s of agent became 7.6 s), and it's squashed unless its size is given. This keeps each
frame's timestamp instead, so the recording is in real time, and `cut_pauses` then takes out the
time the model spent thinking, and the parts of its waits where nothing changes.

Headless Chrome draws no mouse pointer, so `Cursor` draws one into the page and glides it to each
element before the agent clicks or types into it, in place of browser-use's highlight box.

browser-use pins its own google-genai and numpy, so it runs in its own notebook (or virtualenv),
not the one with Chatterbox.
"""

import asyncio
import base64
import contextlib
import json
import os
import shutil
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

from .media import changes, ffmpeg, frame_size

FPS = 30

# The pointer and the ripple of a click, added to every page of the tab. They're in a shadow root, so
# the page's CSS can't restyle them; clicks go through them (pointer-events: none); and browser-use
# doesn't count them as covering what's under them (transparent background, a faint ripple).
CURSOR_JS = """(() => {
  if (window.__cursor) return;
  const ARROW = '<svg xmlns="http://www.w3.org/2000/svg" width="26" height="30" viewBox="0 0 26 30" style="display:block">'
    + '<path d="M3 2v22l6-5.6 4.2 9.4 3.6-1.6-4.1-9.2H21z" fill="#111" stroke="#fff" stroke-width="2" stroke-linejoin="round"/></svg>';
  const fixed = 'position:fixed;left:0;top:0;margin:0;padding:0;pointer-events:none;z-index:2147483647;';
  let pointer = null;
  const place = (x, y, ms) => {
    if (!pointer || !pointer.isConnected) {
      pointer = document.createElement('div');
      pointer.style.cssText = fixed + 'filter:drop-shadow(0 1px 2px rgba(0,0,0,.35))';
      pointer.attachShadow({ mode: 'closed' }).innerHTML = ARROW;
      document.documentElement.appendChild(pointer);
    }
    pointer.style.transition = ms ? `transform ${ms}ms cubic-bezier(.45,0,.2,1)` : 'none';
    pointer.style.transform = `translate(${x - 3}px, ${y - 2}px)`;  // the arrow's tip on (x, y)
  };
  window.__cursor = {
    glide(fromX, fromY, x, y, ms) {
      place(fromX, fromY, 0);
      pointer.getBoundingClientRect();  // start from there, then animate
      place(x, y, ms);
    },
    click(x, y) {
      const ring = document.createElement('div');
      ring.style.cssText = fixed + `left:${x - 22}px;top:${y - 22}px;width:44px;height:44px;border-radius:50%;`
        + 'background:rgba(79,70,229,.45);opacity:.7;transform:scale(.2);transition:transform .45s ease-out,opacity .45s ease-out';
      document.documentElement.appendChild(ring);
      ring.getBoundingClientRect();
      ring.style.transform = 'scale(1.3)';
      ring.style.opacity = '0';
      setTimeout(() => ring.remove(), 500);
    },
  };
})()"""


@dataclass
class Step:
    """One step of the agent's: what it set out to do, what it did, and when (seconds into the video)."""

    goal: str
    actions: list[dict[str, Any]]
    did: list[str]  # browser-use's account of each action, e.g. "Clicked button Generate"
    url: str
    start: float  # the actions start (the model has answered)
    end: float  # they've finished and the page has settled
    clicks: list[list[float]] = field(default_factory=list)  # [when, x, y]: x and y in the video's pixels


class Screencast:
    """Records Chrome's screencast of the tab the agent works in: JPEG frames with their timestamps."""

    def __init__(self, session: Any, frames: Path, size: tuple[int, int]):
        self.session = session
        self.frames = frames
        self.size = size
        self.shots: list[tuple[float, Path]] = []
        self.target: str | None = None  # the CDP session of the recorded tab
        self.stopped = 0.0
        self._acks: set[asyncio.Task] = set()

    @property
    def origin(self) -> float:
        """When the video starts: the first frame's time (Unix seconds)."""
        return self.shots[0][0]

    async def start(self) -> None:
        from browser_use.browser.events import AgentFocusChangedEvent

        self.frames.mkdir(parents=True, exist_ok=True)
        self.session.cdp_client.register.Page.screencastFrame(self._on_frame)
        self.session.event_bus.on(AgentFocusChangedEvent, self._follow_focus)
        await self._follow_focus()

    async def _follow_focus(self, event: Any = None) -> None:
        """Records the tab the agent is in (it can open and switch tabs)."""
        cdp = await self.session.get_or_create_cdp_session()
        if cdp.session_id == self.target:
            return
        if self.target:
            with contextlib.suppress(Exception):  # the old tab may be closed
                await self.session.cdp_client.send.Page.stopScreencast(session_id=self.target)
        self.target = cdp.session_id
        width, height = self.size
        await cdp.cdp_client.send.Page.startScreencast(
            params={"format": "jpeg", "quality": 90, "maxWidth": width, "maxHeight": height},
            session_id=cdp.session_id,
        )

    def _on_frame(self, event: dict[str, Any], session_id: str | None) -> None:
        if session_id != self.target:
            return
        shot = self.frames / f"{len(self.shots):05d}.jpg"
        shot.write_bytes(base64.b64decode(event["data"]))
        self.shots.append((event["metadata"].get("timestamp") or time.time(), shot))
        # Chrome sends the next frame once this one is acknowledged.
        ack = asyncio.ensure_future(self._ack(event["sessionId"], session_id))
        self._acks.add(ack)
        ack.add_done_callback(self._acks.discard)

    async def _ack(self, frame: int, session_id: str | None) -> None:
        with contextlib.suppress(Exception):
            await self.session.cdp_client.send.Page.screencastFrameAck(
                params={"sessionId": frame}, session_id=session_id
            )

    async def stop(self) -> None:
        self.stopped = time.time()
        if self.target:
            with contextlib.suppress(Exception):
                await self.session.cdp_client.send.Page.stopScreencast(session_id=self.target)

    def save(self, out: Path) -> Path:
        """Writes the frames as a 30 fps MP4 in real time (each frame lasts until the next), then deletes them."""
        if not self.shots:
            raise RuntimeError("Chrome sent no screencast frames")
        entries = ["ffconcat version 1.0"]
        untils = [at for at, _ in self.shots[1:]] + [self.stopped]
        for (at, shot), until in zip(self.shots, untils, strict=True):
            entries += [f"file {shot.name}", f"duration {max(until - at, 0.001):.3f}"]
        entries.append(f"file {self.shots[-1][1].name}")  # the last entry's duration is ignored
        listing = self.frames / "frames.ffconcat"
        listing.write_text("\n".join(entries) + "\n")
        out.parent.mkdir(parents=True, exist_ok=True)
        ffmpeg(
            "-f", "concat", "-i", listing,
            "-vf", f"fps={FPS},scale=trunc(iw/2)*2:trunc(ih/2)*2",
            "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", out,
        )  # fmt: skip
        shutil.rmtree(self.frames)
        return out


class Cursor:
    """A mouse pointer drawn into the page, gliding to what the agent clicks or types into."""

    def __init__(self, session: Any, size: tuple[int, int], glide: float = 0.6):
        self.session = session
        self.glide = glide  # seconds
        self.at = (size[0] * 0.6, size[1] * 0.7)
        self.clicks: list[tuple[float, float, float]] = []  # (Unix seconds, x, y), for zooming in on them
        self._tabs: set[str] = set()  # the CDP sessions it's been added to

    async def _run(self, expression: str) -> None:
        cdp = await self.session.get_or_create_cdp_session()
        if cdp.session_id not in self._tabs:  # a new tab: add it to this page and every one it loads
            await cdp.cdp_client.send.Page.addScriptToEvaluateOnNewDocument(
                params={"source": CURSOR_JS}, session_id=cdp.session_id
            )
            await cdp.cdp_client.send.Runtime.evaluate(params={"expression": CURSOR_JS}, session_id=cdp.session_id)
            self._tabs.add(cdp.session_id)
        await cdp.cdp_client.send.Runtime.evaluate(params={"expression": expression}, session_id=cdp.session_id)

    async def show(self) -> None:
        """Draws it where it was (a page the agent just opened doesn't have it yet)."""
        x, y = self.at
        with contextlib.suppress(Exception):  # the page may be in the middle of loading
            await self._run(f"window.__cursor?.glide({x}, {y}, {x}, {y}, 0)")

    async def move(self, x: float, y: float, *, click: bool) -> None:
        (from_x, from_y), self.at = self.at, (x, y)
        await self._run(f"window.__cursor?.glide({from_x}, {from_y}, {x}, {y}, {self.glide * 1000:.0f})")
        await asyncio.sleep(self.glide)
        if click:
            self.clicks.append((time.time(), x, y))
            await self._run(f"window.__cursor?.click({x}, {y})")


async def _target(params: dict[str, Any], session: Any) -> tuple[float, float] | None:
    """Where an action will click or type, in the page's coordinates, if anywhere."""
    if params.get("coordinate_x") is not None and params.get("coordinate_y") is not None:
        return params["coordinate_x"], params["coordinate_y"]
    if params.get("index") is None:
        return None
    node = await session.get_element_by_index(params["index"])
    if node is None:
        return None
    cdp = await session.cdp_client_for_node(node)
    if cdp.session_id != (await session.get_or_create_cdp_session()).session_id:
        return None  # in a cross-site iframe, whose coordinates aren't the page's
    with contextlib.suppress(Exception):  # browser-use scrolls it into view to click it too
        await cdp.cdp_client.send.DOM.scrollIntoViewIfNeeded(
            params={"backendNodeId": node.backend_node_id}, session_id=cdp.session_id
        )
    rect = await session.get_element_coordinates(node.backend_node_id, cdp)
    return (rect.x + rect.width / 2, rect.y + rect.height / 2) if rect else None


def _tools_with(cursor: Cursor) -> Any:
    """browser-use's actions, with the pointer gliding to each one's element first."""
    from browser_use import Tools

    class CursorTools(Tools):
        async def act(self, action: Any, browser_session: Any, **kwargs: Any) -> Any:
            for name, params in action.model_dump(exclude_unset=True).items():
                if params is None:
                    continue
                try:
                    target = await _target(params, browser_session)
                    if target:
                        await cursor.move(*target, click=name in ("click", "input"))
                except Exception:
                    pass  # the pointer is only for show: never fail an action over it
            result = await super().act(action, browser_session, **kwargs)
            await cursor.show()
            return result

    return CursorTools()


def steps_file(video: Path) -> Path:
    return video.with_suffix(".steps.json")


def save_steps(video: Path, steps: list[Step]) -> None:
    steps_file(video).write_text(json.dumps([asdict(step) for step in steps], indent=1, ensure_ascii=False) + "\n")


def load_steps(video: Path) -> list[Step]:
    return [Step(**step) for step in json.loads(steps_file(video).read_text())]


async def record_task(
    task: str,
    out: Path,
    llm: Any,
    *,
    fallback: Any = None,
    url: str | None = None,
    size: tuple[int, int] = (720, 1280),
    max_steps: int = 25,
    storage_state: Path | None = None,
    cookies: list[dict[str, Any]] | None = None,
    cursor: bool = True,
) -> tuple[Path, list[Step]]:
    """Has browser-use's agent do `task`, recording it to `out` (MP4) and its steps to `out.steps.json`.

    Args:
        task: What to do and show, in plain words: "Add three todos, then mark the second one done".
        out: Where to save the MP4.
        llm: browser-use's chat model, e.g. `ChatGoogle(model=..., vertexai=True, project=..., location="global")`.
        fallback: Another model the agent switches to, for the rest of the run, when `llm` answers
            429 or 5xx (Vertex runs out of shared capacity for a model now and then).
        url: The page to start on. It's open before the recording starts, so the video starts on it.
        size: The window, in CSS pixels: 720x1280 gets a site's phone layout. Chrome's screencast
            sends frames at this size whatever the pixel ratio, and `compose` scales them up.
        max_steps: The most steps the agent may take.
        storage_state: Cookies and local storage to start with (a Playwright storage_state JSON), for
            pages behind a sign-in.
        cookies: Cookies to set before `url` opens, as CDP's `Network.CookieParam`s: `signin.signed_in`
            makes makable's. Unlike `storage_state`, which browser-use prints in some warnings, they're
            set straight in Chrome, never logged or saved.
        cursor: Draw a mouse pointer that glides to what the agent clicks, instead of browser-use's
            highlight box.
    """
    from browser_use import Agent, BrowserProfile, BrowserSession

    width, height = size
    profile = BrowserProfile(
        headless=True,
        viewport={"width": width, "height": height},
        chromium_sandbox=os.geteuid() != 0,  # Chrome's sandbox doesn't run as root (Colab)
        keep_alive=True,  # closed below, once the recording has stopped
        storage_state=storage_state,
        highlight_elements=not cursor,  # the box around what it clicks
    )
    session = BrowserSession(browser_profile=profile)
    marks: dict[int, tuple[float, str, list[dict[str, Any]], str]] = {}

    def on_step(state: Any, output: Any, number: int) -> None:  # the model has answered; the actions come next
        actions = [action.model_dump(exclude_none=True) for action in output.action]
        marks[number] = (time.time(), output.next_goal or "", actions, state.url)

    await session.start()
    try:
        if cookies:
            cdp = await session.get_or_create_cdp_session()
            await cdp.cdp_client.send.Storage.setCookies(params={"cookies": cookies}, session_id=cdp.session_id)
        if url:
            await session.navigate_to(url)
            await asyncio.sleep(1.0)  # let it finish drawing
        screencast = Screencast(session, out.parent / f"{out.stem}-frames", size)
        await screencast.start()
        pointer = Cursor(session, size) if cursor else None
        if pointer:
            await pointer.show()
        agent = Agent(
            task=task,
            llm=llm,
            fallback_llm=fallback,
            browser_session=session,
            tools=_tools_with(pointer) if pointer else None,
            register_new_step_callback=on_step,
            directly_open_url=False,  # it's open already
            use_judge=False,  # a second model call that grades the run
            enable_signal_handler=False,  # Jupyter owns Ctrl+C
        )
        history = await agent.run(max_steps=max_steps)
        await asyncio.sleep(1.0)  # hold the end a moment
        await screencast.stop()
    finally:
        await session.kill()

    screencast.save(out)
    steps = []
    for item in history.history:
        if item.metadata is None or item.metadata.step_number not in marks:
            continue
        at, goal, actions, page = marks[item.metadata.step_number]
        # The final report in full: browser-use's memory of it keeps only the first 100 characters.
        did = [
            (result.is_done and result.extracted_content)
            or result.long_term_memory
            or result.extracted_content
            or result.error
            or ""
            for result in item.result
        ]
        clicks = [
            [round(when - screencast.origin, 2), round(x), round(y)]
            for when, x, y in (pointer.clicks if pointer else [])
            if at <= when <= item.metadata.step_end_time
        ]
        steps.append(
            Step(
                goal=goal,
                actions=actions,
                did=did,
                url=page,
                start=round(max(at - screencast.origin, 0.0), 2),
                end=round(item.metadata.step_end_time - screencast.origin, 2),
                clicks=clicks,
            )
        )
    save_steps(out, steps)
    return out, steps


def _waits(step: Step) -> bool:
    """Whether all the step does is wait (say, 30 s while makable builds)."""
    return bool(step.actions) and all(set(action) == {"wait"} for action in step.actions)


def _types(step: Step) -> bool:
    """Whether the step types into the page."""
    return any("input" in action for action in step.actions)


def _zoompan(
    clicks: list[tuple[float, float, float]], zoom: float, size: tuple[int, int], ease: float = 0.4, hold: float = 1.0
) -> str:
    """ffmpeg's zoompan, zooming in `zoom` times on each click (seconds into the video, x, y) and out after.

    It zooms in over the `ease` seconds before a click, stays `hold` seconds, then zooms out over `ease`.
    Clicks closer together than that stay zoomed in, panning from one to the next.
    """

    def eased(begin: float) -> str:  # 0 until `begin`, then smoothly up to 1 at `begin + ease`
        u = f"clip((on/{FPS}-{begin:.3f})/{ease},0,1)"
        return f"{u}*{u}*(3-2*{u})"

    clicks = sorted(clicks)
    groups: list[list[tuple[float, float, float]]] = []
    for click in clicks:
        if groups and click[0] - ease <= groups[-1][-1][0] + hold + ease:
            groups[-1].append(click)
        else:
            groups.append([click])
    level = "+".join(f"{eased(group[0][0] - ease)}-{eased(group[-1][0] + hold)}" for group in groups)
    # What it zooms in on: the first click, then it pans to each next one while zooming in on it.
    pans = list(zip(clicks, clicks[1:], strict=False))
    x = f"{clicks[0][1]:g}" + "".join(f"+({b[1] - a[1]:g})*{eased(b[0] - ease)}" for a, b in pans)
    y = f"{clicks[0][2]:g}" + "".join(f"+({b[2] - a[2]:g})*{eased(b[0] - ease)}" for a, b in pans)
    width, height = size
    # zoompan keeps the window inside the frame. on is the frame's number, so on/FPS its time.
    return f"zoompan=z='1+{zoom - 1:g}*({level})':x='{x}-iw/zoom/2':y='{y}-ih/zoom/2':d=1:s={width}x{height}:fps={FPS}"


def cut_pauses(
    video: Path,
    steps: list[Step],
    out: Path,
    *,
    lead: float = 0.6,
    before: float = 0.5,
    after: float = 1.0,
    typing: float = 4.0,
    zoom: float = 1.5,
) -> tuple[Path, list[Step]]:
    """Cuts out the time the model spent thinking, and the parts of its waits where nothing changes.

    Steps that type play `typing` times as fast: browser-use types a key at a time, so a long prompt
    takes 20 s. The video zooms in `zoom` times on each click, and out after (1: no zooming).

    Keeps each step's actions, from `lead` seconds before them. A step that only waits (say, 30 s while
    makable builds) keeps each change on the page, from `before` seconds before it to `after` seconds
    after: a new line in makable's chat, the preview loading. A wait with nothing new is cut whole.
    makable's ticking timer and shimmering "Thinking…" don't count: they change up to 0.12 % of the
    frame, under `changes`' 0.15 %; a new line in the chat changes 0.3 % or more, the preview 10-60 %.
    Changes while the model thinks are cut too: makable's home page swaps its headline every 2 s.

    Returns the new video and the steps timed for it (also saved as `out.steps.json`).
    """
    waits = [step for step in steps if _waits(step)]
    wanted = [[max(step.start - lead, 0.0), step.end] for step in steps if not _waits(step)]
    wanted += [
        [max(at - before, 0.0), at + after]
        for at in (changes(video, FPS) if waits else [])
        if any(step.start <= at <= step.end for step in waits)
    ]
    spans: list[list[float]] = []
    for begin, end in sorted(wanted):
        if spans and begin <= spans[-1][1]:
            spans[-1][1] = max(spans[-1][1], end)
        else:
            spans.append([begin, end])

    typed = [[step.start, step.end] for step in steps if _types(step)]  # all within `spans`

    def moved(at: float) -> float:  # a time in `video` -> the same moment in `out`
        kept = sum(max(min(end, at) - begin, 0.0) for begin, end in spans)
        fast = sum(max(min(end, at) - begin, 0.0) for begin, end in typed)
        return round(kept - fast * (1 - 1 / typing), 2)

    def between(spans: list[list[float]]) -> str:
        return "+".join(f"between(t,{begin:.3f},{end:.3f})" for begin, end in spans) or "0"

    # While typing, only every `typing`th frame (n is the frame's number in `video`, at FPS).
    keep = f"({between(spans)})*if({between(typed)},lt(mod(n,{typing}),1),1)"
    edit = f"select='{keep}',setpts=N/{FPS}/TB"
    clicks = [(moved(when), x, y) for step in steps for when, x, y in step.clicks]
    if zoom > 1 and clicks:  # in 4:4:4, or zoompan moves in steps of 2 pixels
        edit += f",format=yuv444p,{_zoompan(clicks, zoom, frame_size(video))}"
    ffmpeg(
        "-i", video, "-vf", edit, "-an",
        "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", out,
    )  # fmt: skip
    timed = [
        Step(
            **{
                **asdict(step),
                "start": moved(step.start),
                "end": moved(step.end),
                "clicks": [[moved(when), x, y] for when, x, y in step.clicks],
            }
        )
        for step in steps
    ]
    save_steps(out, timed)
    return out, timed


In [ ]:
%%writefile /content/shorts/signin.py
"""Signs the recorder in to makable as a demo user, with a password instead of GitHub.

makable signs people in with GitHub through Supabase, then keeps Supabase's session in two HttpOnly
cookies (makable-backend's `app/api/cookies.py`). The demo user signed in with GitHub once, so makable
knows it, and also has a password in Supabase. `signed_in` asks Supabase for a session with that
password and gives the browser the same two cookies, so the site opens signed in:

    cookies = signed_in("https://makable.tech", SUPABASE_URL, SUPABASE_KEY, DEMO_EMAIL, DEMO_PASSWORD)
    await record_task(task, out, llm, url="https://makable.tech", cookies=cookies)

Each call starts a new session, so nothing is saved between runs. Sign in again for each recording:
the site rotates the refresh token as it goes, and Supabase ends a session whose old refresh token
comes back.

Password sign-in needs Supabase's Email provider turned on (it answers 422 email_provider_disabled
otherwise).
"""

import json
import time
import urllib.error
import urllib.request
from typing import Any
from urllib.parse import urlsplit

# Each cookie's name and path, as makable-backend sets them (`app/constants/auth.py`).
ACCESS_COOKIE = ("makable_access", "/api/v1")
REFRESH_COOKIE = ("makable_refresh", "/api/v1/auth")
REFRESH_COOKIE_DAYS = 30  # the backend's REFRESH_TOKEN_TTL_DAYS
SESSION_PATH = "/api/v1/auth/session"


def signed_in(app_url: str, supabase_url: str, key: str, email: str, password: str) -> list[dict[str, Any]]:
    """Signs in as `email` and returns the cookies that make `app_url` open signed in, for `record_task`'s
    `cookies`.

    Args:
        app_url: makable's address, e.g. "https://makable.tech".
        supabase_url: https://<ref>.supabase.co
        key: The project's publishable key (or the legacy anon key): the one makable's backend uses.

    Raises:
        RuntimeError: Supabase refused the sign-in, or makable doesn't know the user.
    """
    parts = urlsplit(app_url)
    origin = f"{parts.scheme}://{parts.netloc}"
    session = password_session(supabase_url, key, email, password)
    _check(origin, session["access_token"], email)
    return session_cookies(origin, session)


def password_session(supabase_url: str, key: str, email: str, password: str) -> dict[str, Any]:
    """A new Supabase session for `email`: access_token, refresh_token, expires_in (seconds) and user."""
    # Pasted secrets can end in a line break (Colab kept a "\r\n"), and Supabase matches the email exactly.
    supabase_url, key, email, password = supabase_url.strip(), key.strip(), email.strip(), password.strip("\r\n")
    request = urllib.request.Request(
        f"{supabase_url.rstrip('/')}/auth/v1/token?grant_type=password",
        data=json.dumps({"email": email, "password": password}).encode(),
        headers={"apikey": key, "Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            return json.load(response)
    except urllib.error.HTTPError as err:  # e.g. 400 invalid_credentials, 422 email_provider_disabled
        body = err.read().decode()
        hint = ""
        if "invalid_credentials" in body:
            hint = (
                f"\nSupabase says this when no user in {supabase_url} has the email {email!r}, when that user has "
                "no password yet, or when the password is wrong."
            )
            if password != password.strip():
                hint += " The password starts or ends with a space: check the secret."
        raise RuntimeError(f"Supabase refused the password sign-in ({err.code}): {body}{hint}") from None


def session_cookies(origin: str, session: dict[str, Any]) -> list[dict[str, Any]]:
    """The backend's two auth cookies for `session`, on `origin` (the Worker there forwards /api to the backend),
    as CDP's `Network.CookieParam`s."""
    now = time.time()

    def cookie(name_path: tuple[str, str], value: str, seconds: float) -> dict[str, Any]:
        name, path = name_path
        return {
            "name": name,
            "value": value,
            "url": origin,  # a `url` and no `domain`: only this host gets it, like the backend's own cookie
            "path": path,
            "expires": now + seconds,
            "httpOnly": True,
            "secure": origin.startswith("https://"),
            "sameSite": "Lax",
        }

    return [
        cookie(ACCESS_COOKIE, session["access_token"], session["expires_in"]),
        cookie(REFRESH_COOKIE, session["refresh_token"], REFRESH_COOKIE_DAYS * 86400),
    ]


def _check(origin: str, access_token: str, email: str) -> None:
    """Asks makable who the session belongs to, so a recording never starts on the signed-out page."""
    name, _ = ACCESS_COOKIE
    request = urllib.request.Request(
        origin + SESSION_PATH, headers={"Cookie": f"{name}={access_token}", "User-Agent": "makable-videos"}
    )
    try:
        with urllib.request.urlopen(request, timeout=30):
            return
    except urllib.error.HTTPError as err:
        if err.code == 401:
            raise RuntimeError(
                f"Supabase signed {email} in, but makable doesn't know that user: sign in to makable with the "
                "demo's GitHub account once, and set the password on that same Supabase user"
            ) from None
        raise RuntimeError(f"{origin}{SESSION_PATH} answered {err.code}: {err.read().decode()}") from None


In [ ]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.insert(0, "/content")

from browser_use import ChatGoogle
from IPython.display import Video, display

from shorts.browse import cut_pauses, record_task
from shorts.signin import signed_in


def gemini(model):
    # A timeout, or a stalled request waits forever; and 2 tries, so the fallback takes over soon.
    options = {"http_options": {"timeout": 120_000}, "max_retries": 2}
    if USE_VERTEX:
        return ChatGoogle(model=model, vertexai=True, project=PROJECT, location="global", **options)
    return ChatGoogle(model=model, api_key=userdata.get("GEMINI_API_KEY"), **options)


llm, fallback = gemini(MODEL), gemini(FALLBACK_MODEL)
print("model:", llm.model, "| fallback:", fallback.model)


def show(video, steps):
    display(Video(str(video), embed=True, width=270))
    for step in steps:
        print(f"{step.start:5.1f}-{step.end:5.1f}s  {step.goal}")
        for did in step.did:
            print(f"               {did}")


## 1. Record

Say what to show, in order, the way you'd tell a person. The recording starts with the page in `URL` already open, in a 720×1280 window (the phone layout). Watch the log: each step prints what the model saw and decided.

In [ ]:
URL = "https://demo.playwright.dev/todomvc"
TASK = """
Add three todos, one at a time: "Buy flour", "Bake sourdough" and "Open the shop".
Then mark "Bake sourdough" as done, and show only the active todos.
"""

video, steps = await record_task(TASK, OUT / "todo.mp4", llm, fallback=fallback, url=URL)
show(video, steps)


## 2. Without the model's pauses

In [ ]:
tight, tight_steps = cut_pauses(video, steps, OUT / "todo-tight.mp4")
show(tight, tight_steps)


## 3. makable, signed in as the demo user

browser-use can't do GitHub's sign-in, so the demo user also has a password in Supabase. `signed_in` asks Supabase for a new session with it and gives Chrome the two cookies makable sets after a GitHub sign-in, so the recording opens signed in. It first asks makable who that is, so a wrong setup fails here instead of recording the signed-out page.

It reads four Colab secrets (🔑 in the left bar): `SUPABASE_URL`, `SUPABASE_PUBLISHABLE_KEY`, `MAKABLE_DEMO_EMAIL` and `MAKABLE_DEMO_PASSWORD`.

The agent really uses the site: a site it builds or publishes is made on the demo account's GitHub.

In [ ]:
APP_URL = "https://makable.tech"
TASK = """
You're testing makable, an AI website builder, in a phone-sized window. You're already signed in as a demo user.
Build a new website from start to finish and check that each part works.

1. Start a new site. If the page has a box that says "Describe the site you want to build…", use it.
   If one of your sites opens instead (a chat with a preview of a site under it), first open the menu
   with the site's name at the top of the chat and choose "New site".
   Type this into the box as one line (a line break presses Enter, which sends it), then send it:
   "A one-page website for Crumb & Co, a small neighbourhood bakery: a hero with the bakery's name and a short tagline, a menu of 6 breads and pastries with prices, opening hours, and a contact section with an address and a phone number."

2. makable starts building the site in a chat. If it asks for the site's name, type "Crumb & Co" in the
   chat's box and send it (or click it, if it's one of the suggested names). Answer any other question
   with a short, sensible answer that fits a bakery. If makable says it needs an AI model or an API key,
   stop and report that.

3. Building takes a few minutes. While it works, the chat shows its progress steps and a timer counting
   the seconds. Wait for it: use the wait action for 30 seconds at a time, then look at the page again.
   Don't reload the page, and don't click Stop, Cancel or anything else while it's building.
   Give up if the progress steps haven't moved for 5 minutes, or after 10 minutes in all.

4. When it's done, the chat is at the top and a live preview of the site is under it. Wait until the
   preview's toolbar says "Live preview" (not "Building preview…" or "Preview error"). Check that the
   preview shows the bakery's name, the menu with prices, the opening hours and the contact section.
   Scroll inside the preview to see all of it.

5. Ask for one change in the chat's box ("Ask for a change to your site…"), typed as one line:
   "Make the hero background a warm cream colour and add an 'Order for pickup' button."
   Wait for it the same way (give up after 5 minutes), then check that the preview shows the change.

Don't publish the site, don't change any settings, don't click Undo and don't sign out.

At the end, call done with success true only if every step worked. In its text, report whether each step
worked, anything that looked broken (an error message, a blank or empty preview, a step that never
finished), and roughly how long the build and the change each took (from the chat's timer).
"""


def demo_cookies():  # a new session for each recording (the site rotates its refresh token as it goes)
    return signed_in(
        APP_URL,
        userdata.get("SUPABASE_URL"),
        userdata.get("SUPABASE_PUBLISHABLE_KEY"),
        userdata.get("MAKABLE_DEMO_EMAIL"),
        userdata.get("MAKABLE_DEMO_PASSWORD"),
    )


# Each 30 s wait is a step, and a build takes minutes: 25 steps (the default) runs out before the change.
video, steps = await record_task(
    TASK, OUT / "makable.mp4", llm, fallback=fallback, url=APP_URL, cookies=demo_cookies(), max_steps=60
)
show(video, steps)
tight, tight_steps = cut_pauses(video, steps, OUT / "makable-tight.mp4")
show(tight, tight_steps)
